I initially created my gradients, and then did the study. Problem:
- The separate sum of the components is not equal to a histogram with all components turned on.
My idea: I create gradients for the total, and use them to calculate the off baseline values for each component. I don't think this works.

I retried. Create gradients using all components (also prompt!). And just look at the total, while varying the detector systematics. In table_overlap_BPL_new

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import pickle
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import LogLocator
import os
import numpy as np
from copy import deepcopy

In [4]:
import NNMFit
from NNMFit.utilities.readout_graphs import HistogramGraph
from NNMFit.core.analysis_config import AnalysisConfig

In [5]:
import sys
sys.path.append("/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks")
# from plot_utils import plot_histogram, plot_histogram_astro_flavor, plot_histogram_astro_newflavor
from plot_utils import plot_histogram

from utils import *

In [6]:
plotting_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/bin_migration_overlap_dortmund/table_overlap_BPL_new"
os.system(f"mkdir -p {plotting_path}")

0

First load the configs

In [7]:
input_variables_base = {
    r"All" : {"total_astro_norm" : 3*1.77, "gamma_1" : 1.31, "gamma_2" : 2.74, "e_break" : 4.39,"a" : 0.4444444444, "b" : 0.0,"prompt_norm" : 1, "conv_norm" : 1, "muongun_norm" : 0, "muon_norm" : 0},
}

In [8]:
configs_dir = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/"

config_hdls = {}
hist_graph_hdls = {}
detector_configs = {}

for name, config, config_sys in zip(["clean_gf", "clean_hese", "overlap"],
                                    ["globalfit_hese_clean_globalfit_dortmund", "globalfit_hese_clean_hese_dortmund", "globalfit_hese_overlap_dortmund"],
                                    ["Snowstorm_Gradients_HESE_Globalfit_clean_gf_dortmund", "Snowstorm_Gradients_HESE_Globalfit_clean_hese_dortmund", "Snowstorm_Gradients_HESE_Globalfit_overlap_dortmund"]):

    print(10*"-", name)

    config_hdls[name] = AnalysisConfig.from_configs(
            main_config_file=f"{configs_dir}/main.cfg",
            analysis_config_file = f"{configs_dir}/analysis_configs/asimov/SAY/globalfit_hese/globalfit_double_no_hybrid_hese_BPL_3flavor.yaml",
            config_dir=configs_dir,
            override_dict=None,
            override_config_files=[f"override/systematics/overlap_dortmund/{config_sys}.cfg",
                                   f"override/datasets_GP_globalfit/dortmund/{config}.cfg",
                                   f"override/binning/hese/10bdtprod_threshold_0.122.cfg"],
            override_components_files=["override/components/astro_BPL_3flavor_no_inel.yaml",
                                    "override/muon/muontemplate_hese_11features_plus_rloglmilli_econf_evtgen_bdt1_0.333333_bdt2_0.366667.yaml"],
            override_parameters_files=None)
    hist_graph_hdls[name] = HistogramGraph(config_hdls[name])
    detector_configs[name] = config_hdls[name].get_det_configs()


---------- clean_gf
---------- clean_hese
---------- overlap


Check the rates

In [9]:
input_variables_base = {
    r"All" : {"total_astro_norm" : 3*1.77, "gamma_1" : 1.31, "gamma_2" : 2.74, "e_break" : 4.39,"a" : 0.4444444444, "b" : 0.0,"prompt_norm" : 1, "conv_norm" : 1, "muongun_norm" : 1},
}

# Parameter names
param_names = ["dom_eff", "ice_abs", "ice_scat", "ice_holep0", "ice_holep1", "ice_crystal"]

# Default values
default_values = [1.0, 1.0, 1.0, 0.24901831812365854, -0.05678798504997925, 1.0]

# Build dict of default detector-systematics parameters
default_param_dict = dict(zip(param_names, default_values))

input_variables_nominal_syst = deepcopy(input_variables_base)

for component, param_dict in input_variables_nominal_syst.items():
    param_dict.update(default_param_dict)

Rates match when creating new config hdls, and not setting any of the detector systematics. 

In [10]:
det_configs = ["IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades","IC86_pass2_SnowStorm_v2_cscd_cascade_double"]

rates = {}

for name in ["clean_gf", "clean_hese", "overlap"]:

    print(10*"-", name)

    rates[name] = {}

    for det_config in det_configs:

        print(det_config)

        rates[name][det_config] = get_event_rates(hist_graph_hdl=hist_graph_hdls[name], det_config=det_config, input_variables = input_variables_base)

print("difference", "IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades")
diff = rates["clean_gf"]["IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades"]["All"]["rate"] - rates["clean_hese"]["IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades"]["All"]["rate"]
print(diff, rates["overlap"]["IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades"]["All"]["rate"])
print("difference", "IC86_pass2_SnowStorm_v2_cscd_cascade_double")
diff = rates["clean_hese"]["IC86_pass2_SnowStorm_v2_cscd_cascade_double"]["All"]["rate"] - rates["clean_gf"]["IC86_pass2_SnowStorm_v2_cscd_cascade_double"]["All"]["rate"]
print(diff, rates["overlap"]["IC86_pass2_SnowStorm_v2_cscd_cascade_double"]["All"]["rate"])

---------- clean_gf
IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades
IC86_pass2_SnowStorm_v2_cscd_cascade_double
---------- clean_hese
IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades
IC86_pass2_SnowStorm_v2_cscd_cascade_double
---------- overlap
IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades
IC86_pass2_SnowStorm_v2_cscd_cascade_double
difference IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades
3.123764678962448 3.1237646789624494
difference IC86_pass2_SnowStorm_v2_cscd_cascade_double
2.25184578530113 2.251845785301131


Now I do the same but I set the detector systematics to the nominal. Seems to be fine as well!

In [11]:
det_configs = ["IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades","IC86_pass2_SnowStorm_v2_cscd_cascade_double"]

rates = {}

for name in ["clean_gf", "clean_hese", "overlap"]:

    print(10*"-", name)

    rates[name] = {}

    for det_config in det_configs:

        print(det_config)

        rates[name][det_config] = get_event_rates(hist_graph_hdl=hist_graph_hdls[name], det_config=det_config, input_variables = input_variables_nominal_syst)

print("difference", "IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades")
diff = rates["clean_gf"]["IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades"]["All"]["rate"] - rates["clean_hese"]["IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades"]["All"]["rate"]
print(diff, rates["overlap"]["IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades"]["All"]["rate"])
print("difference", "IC86_pass2_SnowStorm_v2_cscd_cascade_double")
diff = rates["clean_hese"]["IC86_pass2_SnowStorm_v2_cscd_cascade_double"]["All"]["rate"] - rates["clean_gf"]["IC86_pass2_SnowStorm_v2_cscd_cascade_double"]["All"]["rate"]
print(diff, rates["overlap"]["IC86_pass2_SnowStorm_v2_cscd_cascade_double"]["All"]["rate"])

---------- clean_gf
IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades
IC86_pass2_SnowStorm_v2_cscd_cascade_double
---------- clean_hese
IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades
IC86_pass2_SnowStorm_v2_cscd_cascade_double
---------- overlap
IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades
IC86_pass2_SnowStorm_v2_cscd_cascade_double
difference IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades
3.123764678962448 3.1237646789624494
difference IC86_pass2_SnowStorm_v2_cscd_cascade_double
2.25184578530113 2.251845785301131


Now I try to set one variable off the baseline

In [12]:
input_variables_shift_syst = deepcopy(input_variables_nominal_syst)

for component, param_dict in input_variables_shift_syst.items():
    param_dict.update({"dom_eff" : 0.95})

det_configs = ["IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades","IC86_pass2_SnowStorm_v2_cscd_cascade_double"]

rates = {}

for name in ["clean_gf", "clean_hese", "overlap"]:

    print(10*"-", name)

    rates[name] = {}

    for det_config in det_configs:

        print(det_config)

        rates[name][det_config] = get_event_rates(hist_graph_hdl=hist_graph_hdls[name], det_config=det_config, input_variables = input_variables_shift_syst)

print("difference", "IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades")
diff = rates["clean_gf"]["IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades"]["All"]["rate"] - rates["clean_hese"]["IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades"]["All"]["rate"]
print(diff, rates["overlap"]["IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades"]["All"]["rate"])
print("difference", "IC86_pass2_SnowStorm_v2_cscd_cascade_double")
diff = rates["clean_hese"]["IC86_pass2_SnowStorm_v2_cscd_cascade_double"]["All"]["rate"] - rates["clean_gf"]["IC86_pass2_SnowStorm_v2_cscd_cascade_double"]["All"]["rate"]
print(diff, rates["overlap"]["IC86_pass2_SnowStorm_v2_cscd_cascade_double"]["All"]["rate"])

---------- clean_gf
IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades
IC86_pass2_SnowStorm_v2_cscd_cascade_double
---------- clean_hese
IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades
IC86_pass2_SnowStorm_v2_cscd_cascade_double
---------- overlap
IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades
IC86_pass2_SnowStorm_v2_cscd_cascade_double
difference IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades
2.9784996911768107 2.9950960088780683
difference IC86_pass2_SnowStorm_v2_cscd_cascade_double
2.137488404272953 2.1375381678612566


Calculate the rates

In [13]:
det_configs = ["IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades","IC86_pass2_SnowStorm_v2_cscd_cascade_double"]

def calculate_overlap( input_variables ):

    rates = {}

    for name in ["clean_gf", "clean_hese", "overlap"]:

        # print(10*"-", name)

        rates[name] = {}

        for det_config in det_configs: 

            # print(det_config)

            rates[name][det_config] = get_event_rates(hist_graph_hdl=hist_graph_hdls[name], det_config=det_config, input_variables = input_variables)

    # test
    diff = rates["clean_gf"]["IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades"]["All"]["rate"] - rates["clean_hese"]["IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades"]["All"]["rate"]
    overlap = rates["overlap"]["IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades"]["All"]["rate"]
    perc = abs(diff/overlap - 1)
    if perc > 0.05:
        print("difference", "IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades")
        print(diff, overlap, perc)
    
    diff = rates["clean_hese"]["IC86_pass2_SnowStorm_v2_cscd_cascade_double"]["All"]["rate"] - rates["clean_gf"]["IC86_pass2_SnowStorm_v2_cscd_cascade_double"]["All"]["rate"]
    overlap = rates["overlap"]["IC86_pass2_SnowStorm_v2_cscd_cascade_double"]["All"]["rate"]
    perc = abs(diff/overlap - 1)
    if perc > 0.05:
        print("difference", "IC86_pass2_SnowStorm_v2_cscd_cascade_double")
        print(diff, overlap, perc)

    overlap_percentage_hese = rates["overlap"]["IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades"]["All"]["rate"]/rates["clean_gf"]["IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades"]["All"]["rate"]
    overlap_percentage_gf = rates["overlap"]["IC86_pass2_SnowStorm_v2_cscd_cascade_double"]["All"]["rate"]/rates["clean_hese"]["IC86_pass2_SnowStorm_v2_cscd_cascade_double"]["All"]["rate"]

    return rates,overlap_percentage_hese,overlap_percentage_gf


In [16]:
import numpy as np
from copy import deepcopy
from utils import *

# Parameter names
param_names = ["dom_eff", "ice_abs", "ice_scat", "ice_holep0", "ice_holep1", "ice_crystal"]

# Default values
default_values = [1.0, 1.0, 1.0, 0.24901831812365854, -0.05678798504997925, 1.0]

# Ranges for each parameter
param_ranges = [
    (0.9, 1.1),           # dom_eff
    (0.9, 1.1),           # ice_abs
    (0.9, 1.1),           # ice_scat
    (-0.1, 0.5980366362473171),   # ice_holep0
    (-0.1135759700999585, 0.0),   # ice_holep1
    (0.8, 1.2),            # ice_crystal
]

# # half ranges
# param_ranges = [
#     (0.95, 1.05),           # dom_eff
#     (0.95, 1.05),           # ice_abs
#     (0.95, 1.05),           # ice_scat
#     (0.07450915906182926, 0.4235274771854878),   # ice_holep0
#     (-0.08518197757496897, -0.028393992524989625),   # ice_holep1
#     (0.9, 1.1)            # ice_crystal
# ]

# Build dict of default detector-systematics parameters
default_param_dict = dict(zip(param_names, default_values))

n_points = 5  # how many points to sample in each range

for i, param in enumerate(param_names):

    print(10*"-")
    print(param)

    values = np.linspace(param_ranges[i][0],param_ranges[i][1],n_points)

    overlap_hese = []
    overlap_gf = []

    for value in values:

        input_variables = deepcopy(input_variables_base)

        for sample_name in input_variables:
            input_variables[sample_name].update(default_param_dict)
            input_variables[sample_name][param] = value

        _, hese, gf = calculate_overlap(input_variables)

        overlap_hese.append(hese)
        overlap_gf.append(gf)

    # print("overlap_hese", overlap_hese)
    # print("overlap_gf", overlap_gf)

    # -----------------------
    # HESE plot (separate)
    # -----------------------
    plt.figure(figsize=(6, 4))
    plt.plot(values, overlap_hese, marker="o")
    plt.xlabel(param)
    plt.ylabel("Overlap percentage HESE double in GF double")
    plt.title(f"{param} — HESE")
    plt.grid(True)
    plt.tight_layout()

    plt.savefig(f"{plotting_path}/overlap_hese_{param}.png")
    plt.close()

    # -----------------------
    # GF plot (separate)
    # -----------------------
    plt.figure(figsize=(6, 4))
    plt.plot(values, overlap_gf, marker="o")
    plt.xlabel(param)
    plt.ylabel("Overlap percentage GF double in HESE double")
    plt.title(f"{param} — GF")
    plt.grid(True)
    plt.tight_layout()

    plt.savefig(f"{plotting_path}/overlap_gf_{param}.png")
    plt.close()



----------
dom_eff
----------
ice_abs
----------
ice_scat
----------
ice_holep0
----------
ice_holep1
----------
ice_crystal
